In [9]:
from pathlib import Path
from functools import lru_cache
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import accelforge as af
import functools

In [10]:
ARCH = Path("designs/arch.yaml")
WORKLOAD = Path("designs/gpt3_6.7B_kv_cache.yaml")
VALIDATOR = Path("designs/gpt3_175b_kv_cache.yaml")
MAPPING = Path("designs/gpt3_6.7B_kv_cache_mapping.yaml")

In [15]:
TARGET_GENERATED_TOKENS = 128

MAPPER_JOBS = 1

lookaheads_to_test = list(range(4, 132, 4))
token_sweep = list(range(0, 4096 + 128, 512))
output_path = Path("milestone_2_results.csv")

def batch_size_for_lookahead(lookahead: int) -> int:
    # Use one query vector per speculative position under test.
    return max(1, int(lookahead))

af.set_n_parallel_jobs(MAPPER_JOBS)

In [20]:
@functools.cache
def draft_spec(tokens: int): 
    draft_spec = af.Spec.from_yaml(
        ARCH,
        WORKLOAD,
        jinja_parse_data = {
            "BATCH_SIZE": 1, 
            "N_TOKENS": tokens + i, 
            "N_NEW_TOKENS": 1
        }
    )

    # we should ask Michael about this (should we even fuse this in the first place)
    # it might also be a good enough approximation to just the draft spec for the entire lookahead 
    for node in draft_spec.arch.nodes:
        if isinstance(node, af.arch.Memory):
            print(f'Keeping all tensors in {node.name}')
            node.tensors.keep = "All"
            break
    draft_spec.mapper.metrics = af.mapper.Metrics.LATENCY 

    result = draft_spec.map_workload_to_arch()
    return result

@functools.cache
def validator_spec(tokens: int, lookahead: int):
    validator_spec = af.Spec.from_yaml(
        ARCH,
        VALIDATOR,
        # change batch size here 
        jinja_parse_data = {
            "BATCH_SIZE": lookahead, 
            "N_TOKENS": tokens,
            "N_NEW_TOKENS": lookahead + 1
        }
    )
    for node in validator_spec.arch.nodes:
        if isinstance(node, af.arch.Memory):
            node.tensors.keep = "All"
            break
    validator_spec.mapper.metrics = af.mapper.Metrics.LATENCY
    result = validator_spec.map_workload_to_arch()
    return float(result.energy()), float(result.latency())

In [24]:
ARCH = Path("designs/arch.yaml")
WORKLOAD = Path("designs/gpt3_6.7B_kv_cache.yaml")
VALIDATOR = Path("designs/gpt3_175b_kv_cache.yaml")
MAPPING = Path("designs/gpt3_6.7B_kv_cache_mapping.yaml")

BATCH_SIZE = 1
N_TOKENS = 8192
N_NEW_TOKENS = 1
MAPPER_JOBS = 1

def evaluate_workload(tokens : int, lookahead : int): 
    total_energy = 0.0 
    total_latency = 0.0 
    for i in range(lookahead): 
        print(f"Currently validating {i}")
        result = draft_spec(tokens)

        total_energy += float(result.energy())
        total_latency += float(result.latency())
    print("on Validator" )
    validator_spec = af.Spec.from_yaml(
        ARCH,
        VALIDATOR,
        jinja_parse_data = { #validator_spec
            "BATCH_SIZE": lookahead, 
            "N_TOKENS": tokens,
            "N_NEW_TOKENS": lookahead + 1
        }
    )
    for node in validator_spec.arch.nodes:
        if isinstance(node, af.arch.Memory):
            print(f'Keeping all tensors in {node.name}')
            node.tensors.keep = "All"
            break
    validator_spec.mapper.metrics = af.mapper.Metrics.LATENCY 
    result = validator_spec.map_workload_to_arch()

    total_energy += float(result.energy())
    total_latency += float(result.latency())


    return total_energy, total_latency 

def default_kv_cached(tokens: int, lookahead: int): 
    total_energy = 0.0 
    total_latency = 0.0 
    for i in range(lookahead): 
        print(f"Iteration {i}")
        validator_spec = af.Spec.from_yaml(
            ARCH,
            VALIDATOR, 
            jinja_parse_data = {
                "BATCH_SIZE": 1, 
                "N_TOKENS": tokens + i, 
                "N_NEW_TOKENS": 1
            }
        )
        for node in validator_spec.arch.nodes:
            if isinstance(node, af.arch.Memory):
                print(f'Keeping all tensors in {node.name}')
                node.tensors.keep = "All"
                break
        validator_spec.mapper.metrics = af.mapper.Metrics.LATENCY 
        result = validator_spec.map_workload_to_arch()

        total_energy += float(result.energy())
        total_latency += float(result.latency()) 

    return total_energy, total_latency

In [25]:

def simulate_spec_round(gamma: int, alpha: float):
    for i in range(gamma):
        if random.random() >= alpha:
            return i + 1 
    return gamma + 1   

def speculative_decoding(context_len: int, generate_tokens: int,
                         gamma: int, alpha: float):
    total_energy_accum, total_latency_accum = 0.0, 0.0
    tokens_generated = 0
    pos = context_len
    sim_energy, sim_latency = 0.0, 0.0

    while tokens_generated < generate_tokens:
        remaining = generate_tokens - tokens_generated
        effective_gamma = min(gamma, remaining)

        for i in range(effective_gamma):
            e, l = draft_cost(pos + i)
            sim_energy += e
            sim_latency += l

        e, l = validator_cost(pos, effective_gamma)
        sim_energy += e
        sim_latency += l

        accepted = min(simulate_spec_round(effective_gamma, alpha), remaining)
        tokens_generated += accepted
        pos += accepted

    total_energy_accum += sim_energy
    total_latency_accum += sim_latency

    return total_energy_accum, total_latency_accum


In [30]:
context_lengths = list(range(128, 513, 128))
gammas = [3, 6, 9]
results = []

for ctx in context_lengths:
    print(f"Context={ctx}: running baseline...")
    base_energy, base_latency = default_kv_cached(ctx, TARGET_GENERATED_TOKENS)

    for gamma in gammas:
        print(f"  gamma={gamma}")
        spec_energy, spec_latency = speculative_decoding(
            ctx, GENERATE_TOKENS, gamma, ALPHA, N_SIMULATIONS
        )
        results.append({
            "context_len": ctx,
            "gamma": gamma,
            "alpha": ALPHA,
            "base_energy": base_energy,
            "base_latency": base_latency,
            "spec_energy": spec_energy,
            "spec_latency": spec_latency,
            "latency_speedup": base_latency / spec_latency,
            "energy_ratio": spec_energy / base_energy,
        })

df = pd.DataFrame(results)
df.to_csv("spec_decoding_results.csv", index=False)
print("Saved to spec_decoding_results.csv")
df

Context=128: running baseline...
Iteration 0
Keeping all tensors in MainMemory


Getting energy, latency, and leak power for components running each Einsum. : 100%|██████████| 10/10 [00:00<00:00, 42.55it/s]
Generating jobs:   0%|          | 0/10 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum I: 1it [00:00, 334.34it/s]

Generating pmapping templates for compute MAC Einsum I: 0it [00:00, ?it/s]

Generating pmapping templates for compute ScalarUnit Einsum V_new: 0it [00:00, ?it/s]

Generating pmapping templates for compute MAC Einsum V_new: 0it [00:00, ?it/s]
Generating pmapping templates for compute MAC Einsum V_new: 32it [00:00, 197.58it/s]
Generating jobs:  20%|██        | 2/10 [00:00<00:00,  9.68it/s]
Generating pmapping templates for compute ScalarUnit Einsum K_new: 0it [00:00, ?it/s]

Generating pmapping templates for compute MAC Einsum K_new: 0it [00:00, ?it/s]
Generating pmapping templates for compute MAC Einsum K_new: 32it [00:00, 203.13it/s]
Generating jobs:  30%|███       | 3/10 [00:00<00:00,  7.33it/s]
Generating pmapping temp

Einsum I has 1 pmapping job:
	0	[I_in in MainMemory] [I in MainMemory] T-b  T-d  T-m  S-Z-m  S-Z-d  S-Z-b  ScalarUnit computes I
Einsum V_new has 32 pmapping jobs:
	0	[WV in MainMemory] [V_new in MainMemory] [I in MainMemory] T-b  T-d  T-m  S-Z-m  S-Z-h  S-Z-e  S-Z-d  S-Z-b  [I in LocalBuffer] T-e  T-h  [V_new in LocalBuffer] T-d  S-reuse_output-d  S-reuse_input-h  S-reuse_input-e  [WV in Register] T-b  T-m  MAC computes V_new
	1	[WV in MainMemory] [V_new in MainMemory] [I in MainMemory] T-b  T-e  T-h  T-m  S-Z-m  S-Z-h  S-Z-e  S-Z-d  S-Z-b  [V_new in LocalBuffer] T-d  [I in LocalBuffer] T-e  T-h  S-reuse_output-d  S-reuse_input-h  S-reuse_input-e  [WV in Register] T-b  T-m  MAC computes V_new
	2	[WV in MainMemory] [V_new in MainMemory] [I in MainMemory] T-b  T-d  T-m  [I in GlobalBuffer] S-Z-m  S-Z-h  S-Z-e  S-Z-d  S-Z-b  [I in LocalBuffer] T-e  T-h  [V_new in LocalBuffer] T-d  S-reuse_output-d  S-reuse_input-h  S-reuse_input-e  [WV in Register] T-b  T-m  MAC computes V_new
	3	[WV in 

Generating pmappings:   4%|▍         | 12/267 [00:13<04:54,  1.16s/it]


KeyboardInterrupt: 

In [29]:
context_lengths = list(range(640, 1025, 128))
gammas = [3, 6, 9]
results = []

for ctx in context_lengths:
    print(f"Context={ctx}: running baseline...")
    base_energy, base_latency = default_kv_cached(ctx, TARGET_GENERATED_TOKENS)

    for gamma in gammas:
        print(f"  gamma={gamma}")
        spec_energy, spec_latency = speculative_decoding(
            ctx, GENERATE_TOKENS, gamma, ALPHA, N_SIMULATIONS
        )
        results.append({
            "context_len": ctx,
            "gamma": gamma,
            "alpha": ALPHA,
            "base_energy": base_energy,
            "base_latency": base_latency,
            "spec_energy": spec_energy,
            "spec_latency": spec_latency,
            "latency_speedup": base_latency / spec_latency,
            "energy_ratio": spec_energy / base_energy,
        })

df2 = pd.DataFrame(results)
df2.to_csv("spec_decoding_results.csv", index=False)
print("Saved to spec_decoding_results.csv")
df2

Iteration 0
Keeping all tensors in MainMemory


Getting energy, latency, and leak power for components running each Einsum. : 100%|██████████| 10/10 [00:00<00:00, 31.01it/s]
Generating jobs:   0%|          | 0/10 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum I: 1it [00:00, 162.35it/s]

Generating pmapping templates for compute MAC Einsum I: 0it [00:00, ?it/s]

Generating pmapping templates for compute ScalarUnit Einsum V_new: 0it [00:00, ?it/s]

Generating pmapping templates for compute MAC Einsum V_new: 0it [00:00, ?it/s]
Generating pmapping templates for compute MAC Einsum V_new: 32it [00:00, 192.09it/s]
Generating jobs:  20%|██        | 2/10 [00:00<00:00,  8.93it/s]
Generating pmapping templates for compute ScalarUnit Einsum K_new: 0it [00:00, ?it/s]

Generating pmapping templates for compute MAC Einsum K_new: 0it [00:00, ?it/s]
Generating pmapping templates for compute MAC Einsum K_new: 32it [00:00, 198.61it/s]
Generating jobs:  30%|███       | 3/10 [00:00<00:01,  6.98it/s]
Generating pmapping temp

Einsum I has 1 pmapping job:
	0	[I_in in MainMemory] [I in MainMemory] T-b  T-d  T-m  S-Z-m  S-Z-d  S-Z-b  ScalarUnit computes I
Einsum V_new has 32 pmapping jobs:
	0	[WV in MainMemory] [V_new in MainMemory] [I in MainMemory] T-b  T-d  T-m  S-Z-m  S-Z-h  S-Z-e  S-Z-d  S-Z-b  [I in LocalBuffer] T-e  T-h  [V_new in LocalBuffer] T-d  S-reuse_output-d  S-reuse_input-h  S-reuse_input-e  [WV in Register] T-b  T-m  MAC computes V_new
	1	[WV in MainMemory] [V_new in MainMemory] [I in MainMemory] T-b  T-e  T-h  T-m  S-Z-m  S-Z-h  S-Z-e  S-Z-d  S-Z-b  [V_new in LocalBuffer] T-d  [I in LocalBuffer] T-e  T-h  S-reuse_output-d  S-reuse_input-h  S-reuse_input-e  [WV in Register] T-b  T-m  MAC computes V_new
	2	[WV in MainMemory] [V_new in MainMemory] [I in MainMemory] T-b  T-d  T-m  [I in GlobalBuffer] S-Z-m  S-Z-h  S-Z-e  S-Z-d  S-Z-b  [I in LocalBuffer] T-e  T-h  [V_new in LocalBuffer] T-d  S-reuse_output-d  S-reuse_input-h  S-reuse_input-e  [WV in Register] T-b  T-m  MAC computes V_new
	3	[WV in 

Generating pmappings:  24%|██▍       | 64/267 [01:38<05:11,  1.53s/it]


KeyboardInterrupt: 